# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings('ignore')
pd.options.display.max_rows = 10

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
X = df
enrich = pd.read_csv('../data/dayofweek.csv')
y = enrich['dayofweek'].astype('float')
X

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [4]:
def calc_metrics(model) -> tuple:
    out: str = ''
    predict_col = model.predict(X_test)

    acc = accuracy_score(predict_col, y_test)
    precision = precision_score(y_true=y_test, y_pred=predict_col, average='weighted')
    recall = recall_score(y_true=y_test, y_pred=predict_col, average='weighted')
    roc_auc = roc_auc_score(y_true=y_test, y_score=model.predict_proba(X_test), multi_class='ovo', average='weighted')
    
    for name, val in (('accuracy', acc), ('precision', precision), ('recall', recall), ('roc_auc', roc_auc)):
        out += f'{name} is {round(val, 5)}\n'

    return out, (round(acc, 5), round(precision,5), round(recall, 5), round(roc_auc, 5))


In [5]:
svc = SVC(random_state=21, probability=True, C=10, class_weight=None, gamma='auto', kernel='rbf').fit(X_train, y_train)
print(calc_metrics(svc)[0][:-1])

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [6]:
dec_tree = DecisionTreeClassifier(random_state=21, class_weight='balanced', criterion='gini', max_depth=22).fit(X_train, y_train)
print(calc_metrics(dec_tree)[0][:-1])

accuracy is 0.89053
precision is 0.89262
recall is 0.89053
roc_auc is 0.93664


## 4. Random forest

1. The same task for random forest.

In [7]:
rand_forest = RandomForestClassifier(random_state=21, class_weight=None, criterion='gini', max_depth=28, n_estimators=50).fit(X_train, y_train)
print(calc_metrics(rand_forest)[0][:-1])

accuracy is 0.92899
precision is 0.93009
recall is 0.92899
roc_auc is 0.99033


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [8]:
model = RandomForestClassifier(random_state=21, class_weight=None, criterion='gini', max_depth=28, n_estimators=50).fit(X_train, y_train)
model.fit(X_train, y_train)
predict_col = model.predict(X_test)
y_test_compare = y_test.reset_index(drop=True).value_counts().sort_values()
predict_compare = pd.Series(predict_col).value_counts().sort_values()
compare = pd.concat([predict_compare, y_test_compare], keys=['predict', 'as_is'], axis=1)
compare['%_error'] = abs(compare['predict'].values / compare['as_is'].values - 1 )
compare.sort_values(by='%_error', ascending=False)

,predict,as_is,%_error
0.0,23,27,0.148148
1.0,49,55,0.109091
4.0,19,21,0.095238
6.0,76,71,0.070423
5.0,57,54,0.055556
3.0,84,80,0.050000
2.0,30,30,0.000000


In [9]:
y_test_compare = y_test.to_frame()
y_test_compare['predict'] = predict_col

out: dict = {'uid': [], '%_error_count': [], 'row_num': []}
for x in X_test:
    if 'uid' in x:
         user_series = X_test[x].loc[X_test[x] == 1]
         user_y_test = y_test_compare.loc[user_series.index]
         compare = pd.concat([user_series, user_y_test], axis=1)
         compare['%_error'] = abs(compare['predict'].values / compare['dayofweek'].values - 1)
         error_count = compare['%_error'].loc[compare['%_error'] > 0].count() / compare['%_error'].count()
         
         if error_count > 0:
             out['uid'].append(x)
             out['%_error_count'].append(error_count)
             out['row_num'].append(compare.shape[0])

pd.DataFrame(out).sort_values(by='%_error_count', ascending=False)

,uid,%_error_count,row_num
6,uid_user_22,1.000000,1
2,uid_user_16,0.400000,5
15,uid_user_6,0.250000,4
4,uid_user_19,0.210526,19
11,uid_user_3,0.200000,14
...,...,...,...
7,uid_user_24,0.090909,11
0,uid_user_10,0.083333,12
14,uid_user_4,0.074074,27
5,uid_user_2,0.041667,28


In [10]:
out: dict = {'labname': [], '%_error_count': [], 'row_num': []}
for x in X_test:
    if 'labname' in x:
         lab_series = X_test[x].loc[X_test[x] == 1]
         lab_y_test = y_test_compare.loc[lab_series.index]
         compare = pd.concat([lab_series, lab_y_test], axis=1)
         compare['%_error'] = abs(compare['predict'].values / compare['dayofweek'].values - 1)
         error_count = compare['%_error'].loc[compare['%_error'] > 0].count() / compare['%_error'].count()
         
         if error_count > 0:
             out['labname'].append(x)
             out['%_error_count'].append(error_count)
             out['row_num'].append(compare.shape[0])

pd.DataFrame(out).sort_values(by='%_error_count', ascending=False)

,labname,%_error_count,row_num
1,labname_lab03,1.000000,1
2,labname_lab03s,1.000000,1
6,labname_laba06,0.222222,9
4,labname_laba04,0.171429,35
3,labname_lab05s,0.166667,6
7,labname_laba06s,0.133333,15
0,labname_code_rvw,0.076923,13
8,labname_project1,0.054217,186
5,labname_laba05,0.021277,47


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [11]:
def func(models_list: list, params_list: list) -> dict:
    out: dict = {}
    for model, params in zip(models_list, params_list):
        model = model(**params).fit(X_train, y_train)
        out[model] = calc_metrics(model)[0]

    return out


models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
models_params = [{'random_state': 21, 'probability': True, 'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel':'rbf'}, 
                 {'random_state': 21, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22},
                 {'random_state': 21, 'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators':50}]

res = func(models, models_params)

for x in res:
    print(x, res[x], sep='\n')

SVC(C=10, gamma='auto', probability=True, random_state=21)
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878

DecisionTreeClassifier(class_weight='balanced', max_depth=22, random_state=21)
accuracy is 0.89053
precision is 0.89262
recall is 0.89053
roc_auc is 0.93664

RandomForestClassifier(max_depth=28, n_estimators=50, random_state=21)
accuracy is 0.92899
precision is 0.93009
recall is 0.92899
roc_auc is 0.99033

